# without augmentation

In [22]:
import cv2
import os
import random
import numpy as np
import keras
import tensorflow as tf
from keras import Sequential
from keras.layers import Dense,Conv2D,MaxPool2D,Flatten,BatchNormalization,Dropout

In [23]:
train_ds = keras.utils.image_dataset_from_directory(
    directory='datasets/cat_vs_dog/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)
val_ds = keras.utils.image_dataset_from_directory(
    directory='datasets/cat_vs_dog/validation',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)
test_ds = keras.utils.image_dataset_from_directory(
    directory='datasets/cat_vs_dog/test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)

Found 1998 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 612 files belonging to 2 classes.


In [24]:
train_ds

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [25]:
def process(image,label):
  image = tf.cast(image/255. ,tf.float32)
  return image,label

train_ds = train_ds.map(process)
test_ds = test_ds.map(process)
val_ds = val_ds.map(process)

In [26]:
train_ds

<_MapDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [27]:
model= Sequential()

model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2),strides=2,padding='valid'))

model.add(Conv2D(64, kernel_size=(3, 3), activation='relu', input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2),strides=2,padding='valid'))

model.add(Conv2D(128, kernel_size=(3, 3), activation='relu', input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.1))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.1))

model.add(Dense(1, activation='sigmoid'))
model.add(Dropout(0.1))


In [15]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 254, 254, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 125, 125, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 60, 60, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,193 (56.64 MB)

 Trainable params: 14,847,745 (56.64 MB)

 Non-trainable params: 448 (1.75 KB)

In [17]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [18]:
history = model.fit(train_ds,validation_data=val_ds,epochs=10,batch_size=32)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 248s 4s/step - accuracy: 0.5090 - loss: 7.6046 - val_accuracy: 0.5000 - val_loss: 168.5234
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 243s 4s/step - accuracy: 0.4985 - loss: 8.0027 - val_accuracy: 0.5000 - val_loss: 257.0920
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 233s 4s/step - accuracy: 0.5030 - loss: 7.9318 - val_accuracy: 0.5000 - val_loss: 270.4817
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 224s 3s/step - accuracy: 0.5035 - loss: 7.9222 - val_accuracy: 0.5000 - val_loss: 267.0057
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 192s 3s/step - accuracy: 0.4930 - loss: 8.0920 - val_accuracy: 0.5000 - val_loss: 247.4323
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 286s 5s/step - accuracy: 0.4995 - loss: 7.9877 - val_accuracy: 0.5000 - val_loss: 230.2170
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 260s 4s/step - accuracy: 0.5105 - loss: 7.8108 - val_accuracy: 0.5000 - val_loss: 221.9818
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 204s 3s/step - accuracy: 0.4970 - loss: 8.0278 - val_accura

# with augmentation

In [20]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [21]:
bath_size = 16
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    'datasets/cat_vs_dog/train',
    target_size=(256,256),
    batch_size=bath_size,
    class_mode='binary'
)

validation_generator = test_datagen.flow_from_directory(
    'datasets/cat_vs_dog/validation',
    target_size=(256,256),
    batch_size=bath_size,
    class_mode='binary'
)

Found 1998 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [28]:
model= Sequential()

model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2),strides=2,padding='valid'))

model.add(Conv2D(64, kernel_size=(3, 3), activation='relu', input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2),strides=2,padding='valid'))

model.add(Conv2D(128, kernel_size=(3, 3), activation='relu', input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.1))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.1))

model.add(Dense(1, activation='sigmoid'))
model.add(Dropout(0.1))


In [30]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


In [32]:
history = model.fit(
    train_generator,
    steps_per_epoch=100,
    epochs=5,
    validation_data=validation_generator,
    validation_steps=100
)

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 161s 2s/step - accuracy: 0.4969 - loss: 7.9083 - val_accuracy: 0.5000 - val_loss: 124.5646
Epoch 2/5


E:\Deep Learning\tf_env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


100/100 ━━━━━━━━━━━━━━━━━━━━ 58s 573ms/step - accuracy: 0.4575 - loss: 8.6619 - val_accuracy: 0.5000 - val_loss: 201.1441
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 162s 2s/step - accuracy: 0.4794 - loss: 8.2974 - val_accuracy: 0.5000 - val_loss: 206.8768
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 47s 463ms/step - accuracy: 0.5151 - loss: 7.7207 - val_accuracy: 0.5000 - val_loss: 280.2729
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 163s 2s/step - accuracy: 0.5038 - loss: 7.9188 - val_accuracy: 0.5000 - val_loss: 264.7197
